In [2]:
import numpy as np 
import pandas as pd 

In [3]:
food1 = pd.read_csv('/Users/ahana/FoodieFinder/Food_data/food1.csv')
food_2 = pd.read_csv('/Users/ahana/FoodieFinder/Food_data/food_2.csv')

In [4]:
food1.head(1)

,name,ingredients,diet,prep_time,cook_time,flavor_profile,course,state,region
0,Balu shahi,"Maida flour, yogurt, oil, sugar",vegetarian,45,25,sweet,dessert,West Bengal,East


In [5]:
food_2.head(1)

,Unnamed: 0,name,ingredients,diet,prep_time,cook_time,flavor_profile,course,state,region,img_url
0,NaN,Adhirasam,"Rice flour, jaggery, ghee, vegetable oil, elachi",vegetarian,10,50,sweet,dessert,West Bengal,East,https://www.crazymasalafood.com/wp-content/ima...


In [6]:
food_2 = food_2.drop(columns=["Unnamed: 0"])

In [7]:
food = pd.concat([food1, food_2], ignore_index=True)

In [8]:
food.head(4)

,name,ingredients,diet,prep_time,cook_time,flavor_profile,course,state,region,img_url
0,Balu shahi,"Maida flour, yogurt, oil, sugar",vegetarian,45,25,sweet,dessert,West Bengal,East,NaN
1,Boondi,"Gram flour, ghee, sugar",vegetarian,80,30,sweet,dessert,Rajasthan,West,NaN
2,Gajar ka halwa,"Carrots, milk, sugar, ghee, cashews, raisins",vegetarian,15,60,sweet,dessert,Punjab,North,NaN
3,Ghevar,"Flour, ghee, kewra, milk, clarified butter, su...",vegetarian,15,30,sweet,dessert,Rajasthan,West,NaN


In [9]:
food = food.drop_duplicates(subset="name")

In [10]:
food.head().shape


(5, 10)

In [11]:
food.isnull().sum()

name                0
ingredients         0
diet                0
prep_time           0
cook_time           0
flavor_profile      0
course              0
state               0
region              1
img_url           255
dtype: int64

In [12]:
food.dropna().shape

(1, 10)

In [13]:
food["food_id"] = range(1, len(food) + 1)

In [14]:
food=food[['food_id','name','diet','course','flavor_profile','state','ingredients','cook_time','img_url']]

In [15]:
food.head()

,food_id,name,diet,course,flavor_profile,state,ingredients,cook_time,img_url
0,1,Balu shahi,vegetarian,dessert,sweet,West Bengal,"Maida flour, yogurt, oil, sugar",25,NaN
1,2,Boondi,vegetarian,dessert,sweet,Rajasthan,"Gram flour, ghee, sugar",30,NaN
2,3,Gajar ka halwa,vegetarian,dessert,sweet,Punjab,"Carrots, milk, sugar, ghee, cashews, raisins",60,NaN
3,4,Ghevar,vegetarian,dessert,sweet,Rajasthan,"Flour, ghee, kewra, milk, clarified butter, su...",30,NaN
4,5,Gulab jamun,vegetarian,dessert,sweet,West Bengal,"Milk powder, plain flour, baking powder, ghee,...",40,NaN


In [16]:
food['tags'] = food['flavor_profile'] + " " + food['course'] + " " + food['state']+ " " + food['diet']

In [17]:
new_df = food[['food_id','name','tags','ingredients','cook_time','img_url']]

In [18]:
new_df.head(2)

,food_id,name,tags,ingredients,cook_time,img_url
0,1,Balu shahi,sweet dessert West Bengal vegetarian,"Maida flour, yogurt, oil, sugar",25,NaN
1,2,Boondi,sweet dessert Rajasthan vegetarian,"Gram flour, ghee, sugar",30,NaN


VECTORIZATION

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
# ✅ NEW — use ingredients + name + tags together for precise matching
import re

def build_soup(row):
    name_clean = row['name'].lower().replace(' ', '_')              # e.g. masala_dosa
    ingredients = re.sub(r'[^a-zA-Z, ]', '', str(row['ingredients']))
    tags = str(row['tags']).replace('-1', '').strip()              # remove -1 placeholder
    # Repeat name 3x so dish identity weighs heavily
    return f"{name_clean} {name_clean} {name_clean} {ingredients} {tags}"

new_df['soup'] = new_df.apply(build_soup, axis=1)

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(new_df['soup']).toarray()
similarity = cosine_similarity(vectors)

In [35]:
similarity = cosine_similarity(vectors)

In [36]:
similarity.shape

(256, 256)

In [37]:
import pickle

In [38]:
new_df.to_csv("new_df.csv", index=False)

In [39]:
pickle.dump(new_df.to_dict(),open('food_dict.pkl','wb'))

In [40]:
pickle.dump(similarity,open('similarity.pkl','wb'))

In [44]:
recommend('Chhena poda')

⚠️  No close matches for 'Chhena poda' (diet=veg, family=other).
    Try lowering threshold, e.g. recommend('Chhena poda', threshold=0.3)
